In [37]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import importlib

In [38]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [39]:
compact = True

In [40]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading 2024-08-30_22-21-59.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4843967728,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.45420,0.013626,0,"[7, 1, 0, 0, 0]"
6139257088,4.843968e+09,[],"[96, 91]",0,"[4, 9]","[4, 9]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,5,0.45420,0.029523,0,"[7, 1, 0, 0, 0]"
4865770928,6.139257e+09,"[24, 20, 4]","[91, 91]",0,"[0, 0]","[9, 9]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,9,0.71980,0.064782,1,"[7, 11, 4, 1, 0]"
4843775744,4.865771e+09,"[24, 20, 4, 50]","[82, 82]",0,"[0, 0]","[18, 18]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,18,0.73890,0.133002,2,"[11, 7, 4, 0, 0]"
6141055568,NaN,[],"[114, 78]",0,"[4, 4]","[4, 4]","[False, True]","[False, False]",1,4,Tord,HumanPlayer,raise,4,0.64511,0.025804,0,"[12, 11, 0, 0, 0]"
6141056816,6.141056e+09,"[13, 33, 44]","[110, 68]",0,"[0, 6]","[8, 14]","[False, True]","[False, False]",1,4,Tord,HumanPlayer,call,6,0.49160,0.054076,0,"[12, 11, 7, 5, 0]"
6164859632,6.141057e+09,"[13, 33, 44, 36]","[104, 62]",0,"[0, 6]","[14, 20]","[False, True]","[False, False]",1,4,Tord,HumanPlayer,call,6,0.49970,0.084949,0,"[12, 11, 10, 7, 5]"
6164728848,6.164860e+09,"[13, 33, 44, 36, 46]","[98, 62]",0,"[0, 0]","[20, 20]","[False, True]","[False, False]",1,4,Tord,HumanPlayer,check,0,0.50390,0.100780,1,"[7, 12, 11, 10, 0]"
6141058688,NaN,[],"[136, 58]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.42280,0.012684,0,"[3, 0, 0, 0, 0]"


In [41]:
raw_df.dtypes

prev_entry           float64
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

## Process data

In [42]:
df = Processor(raw_df).get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,rank,p,relative_ev,stage
4843967728,f160f3c2-114b-47e4-bcf9-6e07453212d2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,2,0,0.45420,0.013626,preflop
6139257088,f160f3c2-114b-47e4-bcf9-6e07453212d2,0,0,0,0,0,2,0,0,0,...,0,0,0,0,call,5,0,0.45420,0.029523,preflop
4865770928,f160f3c2-114b-47e4-bcf9-6e07453212d2,0,0,0,0,0,7,0,0,0,...,1,0,0,0,raise,9,1,0.71980,0.064782,flop
4843775744,f160f3c2-114b-47e4-bcf9-6e07453212d2,0,9,0,0,0,7,0,0,0,...,1,0,0,0,raise,18,2,0.73890,0.133002,turn
6141055568,2c66876c-0db8-4f5f-8738-fa06fcc7bfff,0,0,0,0,0,0,0,0,0,...,0,0,0,0,raise,4,0,0.64511,0.025804,preflop
6141056816,2c66876c-0db8-4f5f-8738-fa06fcc7bfff,4,0,0,0,0,0,0,0,0,...,0,0,0,0,call,6,0,0.49160,0.054076,flop
6164859632,2c66876c-0db8-4f5f-8738-fa06fcc7bfff,4,0,0,0,0,0,6,0,0,...,0,0,0,0,call,6,0,0.49970,0.084949,turn
6164728848,2c66876c-0db8-4f5f-8738-fa06fcc7bfff,4,0,0,0,0,0,6,6,0,...,0,0,1,0,check,0,1,0.50390,0.100780,river
6141058688,6fb84267-d73c-47ff-94a3-2c604afd505d,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,2,0,0.42280,0.012684,preflop
6164728608,6fb84267-d73c-47ff-94a3-2c604afd505d,0,0,0,0,0,2,0,0,0,...,0,0,0,0,call,6,0,0.42280,0.029596,preflop


In [43]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 